# 1. Business Understanding🌱 Crops Yield & Price Forecast | Manuel Lombardi

El contexto del problema a resolver se desarrolla en el área agrícola de la República Argentina. Muchas veces, al terminar una campaña de cosecha, el productor agropecuario tiende a plantearse si acopiar el cultivo cosechado, venderlo o acopiar una parte y vender otra, esta duda muchas veces nace a partir de la volatilidad de los precios de cada cultivo y cómo pueden llegar a cambiar si lo vende hoy, en 15, 30, 40 o 90 días por ejemplo. Otra duda que suele aparecer al inicio de la campaña, que sería al momento de la siembra, es cuándo debe cosechar lo sembrado, si a los 2, 3 o 4 meses, esta duda se mantiene a lo largo de la campaña ya que muchas decisiones se toman en base a cómo va evolucionando el clima en este período de campaña. Mediante el estudio de los datos presentes en este caso, el análisis y luego la creación de modelos predictivos para intentar predecir cuándo realizar cada una de estas acciones, se buscará responder a estas preguntas que surgen tanto en un pequeño productor, como en una gran empresa.

## 1.1 Relevancia


Lo relevante de este trabajo, es la ganancia que se puede ayudar a conseguir a los productores, empresas grandes tanto agrícolas como acopiadoras y también ayudar a la organización de recursos, herramientas y esfuerzos del equipo para planificar cuándo se va a cosechar en base a información climática histórica.

## 1.2 Stakeholders

Los Stakeholders o personas que se verían afectadas por la implementación de estos modelos predictivos serían, entre otros:  
- Productores agropecuarios  
- Empresas de acopio de cereales  
- Financierias que ayudan en el sector agropecuario  
- Semilleros  
- Maquinistas independientes  
- Ingenieros agrónomos  
- Empresas vendedoras de insumos agrícolas

## 1.3 Preguntas a responder

Mediante el análisis de los datos de precios históricos de Maíz, Soja, Girasol y Trigo se buscará responder:  
**¿Qué precio tendrá un cultivo de acá a 15, 30, 60 o 90 días?**

Mediante el análisis y estudio de datos metereológicos de las distintas zonas del país y datos de rendimientos de campañas anteriores, se buscará responder:  
**En base a las condiciones climáticas de la campaña actual ¿qué rendimiento se espera (qq/ha)?**

## 1.4 Objetivo del trabajo

Brindar una solución que, mediante un pronóstico de precio confiable,
ayude al productor a decidir cuándo vender en un plazo de 15 a 90 días;
y una estimación del rendimiento esperado en qq/ha en base a las condiciones
climáticas de la campaña actual.

Para ello se trabajará con datos de campañas desde el año 2000, abarcando
4 cultivos (Maní, Soja, Maíz, Girasol y Trigo) en distintos departamentos de la
República Argentina definidas, complementados con datos meteorológicos obtenidos mediante la API de NASA POWER, índices NDVI y precios históricos de mercado.

##1.5 Preguntas analíticas para guiar los siguientes pasos

Para guiar las etapas que vienen a continuación, establecemos preguntas que buscaremos responder con ayuda del EDA y así poder entender mejor el dataset.


- ¿Cuáles han sido las mejores campañas de cada cultivo a lo largo de la historia en cada zona del país?
- ¿Qué tanto ha ido variando el precio de cada cultivo?
- ¿Cuál es el cultivo más caro a de los cuatro que estudiaremos?
- ¿Hay mucha diferencia en cuanto a la cantidad sembrada de cada cultivo?

## 1.6 Criterios de éxito

Para determinar si los modelos desarrollados son útiles y justifican su complejidad,
se definen criterios de éxito tanto desde la perspectiva del negocio como desde
la perspectiva técnica.

### Criterios de éxito del negocio

- El productor puede consultar el precio esperado de su cultivo a 30, 60 y 90 días
y tomar una decisión de venta fundamentada en datos.
- El productor puede estimar el rendimiento esperado en qq/ha para su zona
antes o durante la campaña, en función de las condiciones climáticas.

### Criterios de éxito del modelo

| Métrica | Track | Umbral mínimo | Qué significa si no se cumple |
|---|---|---|---|
| MAPE | Precio | ≤ 15% a 30 días | El modelo no mejora la intuición del mercado |
| Mejora vs. baseline naive | Precio | ≥ 20% de reducción de error vs. random walk | El modelo no justifica su complejidad |
| MAE | Rendimiento | ≤ 5 qq/ha | El error supera la diferencia entre una buena y mala decisión de siembra |
| R² | Rendimiento | ≥ 0.65 | El modelo no captura suficiente variabilidad climática y zonal |
| Mejora vs. baseline naive | Rendimiento | ≥ 20% de reducción de MAE vs. promedio histórico por zona | El modelo no agrega valor sobre el historial simple |

> ⚠️ **Nota metodológica:** los umbrales definidos en esta tabla son preliminares.
> Serán revisados y ajustados al finalizar la fase de Data Understanding (notebook 02),
> una vez analizada la variabilidad real de precios y rendimientos en el dataset.
> Esta práctica es consistente con el marco CRISP-DM, que permite iterar entre fases.

## 1.7 Restricciones y supuestos



### Restricciones

- **Agregación de celdas climáticas por zona:** la API de NASA POWER mediante
  el endpoint `/regional` devuelve una grilla de celdas de aproximadamente
  50 km de resolución dentro del bounding box definido para cada zona. Dado
  que las zonas no son rectángulos perfectos, el bounding box puede incluir
  celdas que geográficamente caen fuera del límite real de la zona. Para
  obtener un único valor diario por zona se utilizará promedio simple de todas
  las celdas dentro del bounding box, lo cual es una aproximación válida pero
  no pondera por superficie sembrada real.

- **Datos de rendimiento agregados a nivel zonal:** el dataset de la Bolsa
  de Cereales de Buenos Aires reporta rendimiento promedio por zona, no por
  lote ni partido. El modelo no podrá predecir rendimiento a nivel de campo
  individual, sino estimaciones zonales.

- **Cebada y Sorgo excluidos del modelo principal:** ambos cultivos cuentan
  con menos de 18 campañas de histórico, lo cual es insuficiente para entrenar
  modelos robustos. Quedan documentados como trabajo futuro.

### Supuestos

- **Los patrones históricos tienen validez predictiva:** se asume que las
  relaciones entre clima y rendimiento observadas en campañas pasadas son
  representativas de campañas futuras. Eventos estructurales de cambio
  climático de largo plazo podrían invalidar este supuesto.

- **El precio histórico captura suficiente señal por sí solo:** el modelo
  de precio no incorpora variables macroeconómicas como tipo de cambio,
  retenciones o intervención estatal. Se asume que el histórico de precios
  refleja implícitamente el efecto acumulado de estos factores, aunque
  shocks políticos abruptos como cambios de retenciones pueden generar
  errores no capturables por el modelo.

- **El promedio simple de celdas es representativo de cada zona:** se asume
  que promediar todas las celdas dentro del bounding box es suficientemente
  válido para capturar el comportamiento climático predominante de cada zona,
  sin necesidad de ponderar por superficie sembrada.